In [5]:
import os
import sys
from pathlib import Path
import librosa
import soundfile as sf
from tqdm import tqdm

In [2]:
input_path = Path('/home/isol/work/data/raw/temp')
output_path = Path('/home/isol/work/data/raw/temp_cut')
audio_files = list(input_path.glob("*.wav")) + list(input_path.glob("*.mp3")) + list(input_path.glob("*.ogg")) + list(input_path.glob("*.flac"))

print(f"\n✓ 발견된 파일: {len(audio_files)}개")


✓ 발견된 파일: 15개


In [3]:
def split_audio_file(input_file, output_dir, duration=1.0, sample_rate=16000):
    """
    음성 파일을 지정된 길이로 분할
    
    Args:
        input_file: 입력 파일 경로
        output_dir: 출력 디렉토리
        duration: 분할 길이 (초)
        sample_rate: 샘플레이트
    
    Returns:
        분할된 파일 개수
    """
    try:
        # 음성 로드
        audio, sr = librosa.load(input_file, sr=sample_rate)

        # 원본 파일명 (확장자 제외)
        filename_stem = Path(input_file).stem

        # 분할 샘플 수
        samples_per_chunk = int(duration * sample_rate)

        # 분할된 파일 개수
        num_chunks = (len(audio) + samples_per_chunk - 1) // samples_per_chunk

        # 분할 및 저장
        for i in range(num_chunks):
            start_idx = i * samples_per_chunk
            end_idx = min((i + 1) * samples_per_chunk, len(audio))

            chunk = audio[start_idx:end_idx]

            if len(chunk) == samples_per_chunk:
                # 파일명: Car_1_0001.wav
                output_filename = f"{filename_stem}_{i+1:04d}.wav"
                output_file = output_dir / output_filename

                # 저장
                sf.write(output_file, chunk, sample_rate)
        
        return num_chunks

    except Exception as e:
        print(f"\n 처리 실패: {input_file}")
        print(f"   에러: {e}")
        return 0

In [4]:
total_chunks = 0

for input_file in tqdm(audio_files, desc="분할 진행"):
    num_chunks = split_audio_file(
        input_file,
        output_path,
    )
    total_chunks += num_chunks

분할 진행: 100%|██████████| 15/15 [07:11<00:00, 28.74s/it]


In [52]:
len(os.listdir("/home/isol/work/data/raw/temp_cut"))

59472